# Import

In [25]:
import os
import glob
import numpy as np
import graphical_sampling as gs
import pandas as pd

# NHT Variance

In [27]:
N = 100
n = 4
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
variable = np.random.rand(N) * 10
# inclusions = rng.unequal_inclusions(n, N)
inclusions = variable + np.random.rand(N)
pop = gs.Population(coords, inclusions, n=n, variable=variable)

In [37]:
df = pd.read_csv('populations/MU284_filtered.csv')
cs82 = df['CS82'].values.astype(float)
rmt85 = df['RMT85'].values.astype(float)
ss82 = df['SS82'].values.astype(float)

N = len(df)
n = 10
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
pop = gs.Population(
    coords=rng.rand_coord((N, 2)),
    inclusions=ss82,
    variable=cs82,
    n=4
)

In [41]:
initial_designs = [gs.Design(pop, num_zones=num_zones) for _ in range(100)]
criteria = gs.criteria.VarNHT()
gbfs = gs.search.GreedyBestFirstSearch(initial_designs, criteria)

In [42]:
gbfs.run(
    max_iterations=1000,
    max_open_set_size=2000,
    top_k=10,
    num_new_order_nodes=0,
    num_new_exchange_nodes=5,
    num_clusters=2,
    num_zones=1,
    num_changes=2,
    num_zone_changes=1,
    random_pull=True,
    exchange_coef=1.0
)

--- Starting GBFS: Max Iterations=1000, Initial Designs=100 ---
Initial best criteria value: 521557.7660
Iter     0/1000 | Best: 521557.7660 | Open:   100 | Closed:     0
  [!] New best found at iter 0: 521430.7479 (improved by 127.0181)
  [!] New best found at iter 0: 521005.6183 (improved by 425.1296)
  [!] New best found at iter 1: 520938.0065 (improved by 67.6118)
  [!] New best found at iter 2: 520620.1828 (improved by 317.8238)
  [!] New best found at iter 2: 520445.4483 (improved by 174.7345)
  [!] New best found at iter 4: 519779.2618 (improved by 666.1866)
  [!] New best found at iter 5: 519677.7940 (improved by 101.4678)
  [!] New best found at iter 5: 515266.1849 (improved by 4411.6091)
  [!] New best found at iter 6: 514966.4876 (improved by 299.6973)
  [!] New best found at iter 7: 514896.4464 (improved by 70.0412)
  [!] New best found at iter 7: 514845.1543 (improved by 51.2921)
  [!] New best found at iter 8: 514715.6884 (improved by 129.4660)
  [!] New best found at ite

# Spread

In [ ]:
N = 100
n = 4
num_zones = 4
rng = gs.random.rng()
coords = rng.rand_coord((N, 2))
variable = np.random.rand(N) * 10
# inclusions = rng.unequal_inclusions(n, N)
inclusions = variable + np.random.rand(N)
pop = gs.Population(coords, inclusions, n=n, variable=variable)

In [43]:
DATA_DIR = "populations"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))

coords_dict = {}
probs_dict = {}

for fp in csv_paths:
    name = os.path.splitext(os.path.basename(fp))[0]
    data = np.loadtxt(fp, delimiter=",", skiprows=1)
    coords = data[:, :2]
    probs  = data[:, -1]

    coord_name, prob_name, *rest = name.split("_")
    coord_name = 'cluster' if coord_name == 'clust' else coord_name
    prob_name = 'equal' if prob_name == 'eq' else 'unequal'

    coords_dict[coord_name] = coords
    probs_dict[coord_name] = probs_dict.get(coord_name, {})
    probs_dict[coord_name][prob_name] = probs

print(coords_dict.keys())

dict_keys(['swiss', 'RegularPop1000', 'AggregatedPop1027', 'cluster', 'meuse', 'random', 'grid', 'MU284'])


In [45]:
N = 100
n = 5
coords = coords_dict['random']
probs = probs_dict['random']['unequal']
pop = gs.Population(coords, probs, n=n)

In [51]:
initial_designs = []
for _ in range(20):
    fbn = gs.clustering.FIPBalancedNMeans(n)
    fbn.fit(pop)
    design = gs.Design(pop, num_zones=n, order=gs.Order.from_clusters(pop, fbn.clusters))
    initial_designs.append(design)

In [52]:
criteria = gs.criteria.MoranCriteria()
gbfs = gs.search.GreedyBestFirstSearch(initial_designs, criteria)

In [53]:
gbfs.run(
    max_iterations=1000,
    max_open_set_size=1000,
    top_k=10,
    num_new_order_nodes=5,
    num_new_exchange_nodes=5,
    num_clusters=2,
    num_zones=1,
    num_changes=2,
    num_zone_changes=1,
    random_pull=True,
    exchange_coef=1.0
)

--- Starting GBFS: Max Iterations=1000, Initial Designs=20 ---
Initial best criteria value: -0.2976
Iter     0/1000 | Best: -0.2976 | Open:    20 | Closed:     0
  [!] New best found at iter 1: -0.2976 (improved by 0.0000)
  [!] New best found at iter 14: -0.2978 (improved by 0.0003)
  [!] New best found at iter 16: -0.2986 (improved by 0.0008)
  [!] New best found at iter 19: -0.2991 (improved by 0.0005)
  [!] New best found at iter 70: -0.2992 (improved by 0.0002)
  [!] New best found at iter 72: -0.2994 (improved by 0.0001)
  [!] New best found at iter 72: -0.2994 (improved by 0.0001)
  [!] New best found at iter 73: -0.2994 (improved by 0.0000)
  [!] New best found at iter 77: -0.2996 (improved by 0.0002)
  [!] New best found at iter 79: -0.3002 (improved by 0.0006)
Iter   100/1000 | Best: -0.3002 | Open:   410 | Closed:    98
  [!] New best found at iter 129: -0.3003 (improved by 0.0001)
  [!] New best found at iter 133: -0.3007 (improved by 0.0003)
  [!] New best found at iter 13